# 🏥 ShasthoSetu AI — DermoPed Controlled Research Pipeline (v2.3)
### Experiment D: Backbone Freezing + Balanced Sampling (Anti-Overfitting Benchmark)
---
**Controlled Comparison Context (SRS Section 16.2 & 21):**
- **EXP_A (Baseline v1):** Weighted Loss, fully trainable backbone -> Val F1: 48.37%, severe overfitting.
- **EXP_B (Controlled v2):** WeightedRandomSampler, fully trainable backbone -> Val F1: 57.76% (+9.4% F1), Pediculosis Val recall tripled (5% -> 15%), but overfitting persisted (Val loss 1.95).
- **EXP_D (Current Benchmark):** WeightedRandomSampler + Backbone Freezing (`conv1..layer3` frozen, BatchNorm eval) + Dropout 0.5 + Early Stopping. Specifically engineered to stop the 11M parameter network from memorizing the 1 single Pediculosis training patient.

> **Review note (v2.3.1):** A class-prefixed folder ID does not establish global patient identity across conditions. Report all patient counts and disclose the one-patient minority-class limitation. Public proof bundles omit original patient images by default. Do not select a model using the reused held-out test results.


In [ ]:
# =====================================================================
# Cell 1: Environment, Experiment Configuration & Hardware Timer
# =====================================================================
import os, sys, re, json, time, math, random, hashlib, shutil, zipfile, platform
from pathlib import Path
from datetime import datetime, timezone
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torchvision
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

# --- CONTROLLED EXPERIMENT CONFIGURATION ---
EXPERIMENT_MODE = 'EXP_D'  # EXP_D: Backbone Freezing + Balanced Sampler + Dropout 0.5
TIER = 1                  # Tier 1: chickenpox vs pediculosis
SEED = 42
EPOCHS = 10
BATCH_SIZE = 16
MAX_CROPS_PER_PATIENT = 40
EARLY_STOPPING_PATIENCE = 3
SAVE_PROOFS_TO_DRIVE = False  # Standalone Colab: no Google Drive popup
INCLUDE_PATIENT_IMAGES_IN_PUBLIC_ZIP = False  # Default privacy-safe: original clinical images are NOT distributed

EXP_ID = f'DERMOPED-T{TIER}-V2_3_1-{EXPERIMENT_MODE}'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
print('=' * 75)
print(f'STARTING CONTROLLED BENCHMARK: {EXP_ID}')
print('=' * 75)

BASE = Path('/content/shastho_dermoped') if Path('/content').exists() else Path.cwd() / 'shastho_dermoped'
DATA_ROOT = BASE / 'dataset'
ARCHIVE_ROOT = BASE / 'archives'
OUT = BASE / 'evidence' / EXP_ID / RUN_ID
if SAVE_PROOFS_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = Path('/content/drive/MyDrive/ShasthoSetu_Research') / EXP_ID / RUN_ID
for p in [DATA_ROOT, ARCHIVE_ROOT, OUT]: p.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Compute Device   : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU Hardware     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM Total       : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
    x = torch.randn(16, 16, device=DEVICE, requires_grad=True)
    (x.square().mean()).backward()
    assert x.grad is not None
    print('CUDA Smoke Test  : PASS')
else:
    print('⚠️ Running on CPU! For fast training, enable T4 GPU in Runtime settings.')

### Step 2: Download with Exact Stopwatch Timing & Checksum Verification

In [ ]:
# =====================================================================
# Cell 2: Ingestion with Stopwatch Timing & Verified Checksums
# =====================================================================
CATALOG = {
    'chickenpox': ('chickenpox.zip', '6f4de33ee140aab91ea626eb860a2544', 1),
    'pediculosis': ('pediculosis.zip', '811b59d0589b16f6cd7bb146f14e9d51', 1),
    'morbilliform_exanthema': ('morbilliform_exanthema.zip', 'c134746745c531112229db69c0750093', 2),
    'polymorphous_exanthema': ('polymorphous_exanthema.zip', '0e0a79981294a538cedc92943e45bf88', 2),
    'maculopapular_exanthema': ('maculopapular_exanthema.zip', 'fbc0d3adae7f9b4c6878cae8515aeebf', 3),
    'iatrogenic_drug_induced_exanthema': ('iatrogenic_drug_induced_exanthema.zip', '4be24b1dd1dee6c930321ed8038531a7', 3),
    'scabies': ('scabies.zip', '2b75d20fc0886746855b9890185c7525', 4),
    'viral_exanthema': ('viral_exanthema.zip', '7dbce24d6c4f11a9f4ad21eb45656ad1', 4),
    'urticaria': ('urticaria.zip', 'af4bcdc63e55da0faf64d0aa17954a81', 4)
}

SELECTED = [name for name, (_, _, tier) in CATALOG.items() if tier <= TIER]
print(f'Selected Classes ({len(SELECTED)}): {SELECTED}')

def md5sum(path, chunk=2**20):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for data in iter(lambda: f.read(chunk), b''): h.update(data)
    return h.hexdigest()

download_telemetry = {}

def download_timed(filename, expected_md5):
    dest = ARCHIVE_ROOT / filename
    if dest.exists() and md5sum(dest) == expected_md5:
        print(f'✅ Found existing valid archive: {filename}')
        download_telemetry[filename] = {'status': 'cached', 'elapsed_sec': 0.0, 'md5': expected_md5}
        return dest
        
    if dest.exists(): dest.unlink()
    url = f'https://zenodo.org/records/20829589/files/{filename}?download=1'
    tmp = dest.with_suffix('.download')
    
    print(f'⬇️ Downloading {filename} from Zenodo...')
    t0 = time.perf_counter()
    retries = 3
    for attempt in range(1, retries + 1):
        try:
            res = os.system(f'wget -c "{url}" -O "{tmp}" -q --show-progress')
            if res != 0 or not tmp.exists() or tmp.stat().st_size < 1024*1024:
                with requests.get(url, stream=True, timeout=(30, 120)) as r:
                    r.raise_for_status()
                    with open(tmp, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=2**20):
                            if chunk: f.write(chunk)
            
            actual_md5 = md5sum(tmp)
            if actual_md5 != expected_md5:
                raise ValueError(f'MD5 mismatch: expected {expected_md5}, got {actual_md5}')
            
            t_elapsed = round(time.perf_counter() - t0, 2)
            tmp.replace(dest)
            file_mb = round(dest.stat().st_size / (1024**2), 2)
            speed_mbps = round(file_mb / max(t_elapsed, 0.01), 2)
            print(f'✅ {filename} ({file_mb} MB) downloaded in {t_elapsed}s (~{speed_mbps} MB/s) | MD5 verified.')
            download_telemetry[filename] = {'status': 'downloaded', 'elapsed_sec': t_elapsed, 'size_mb': file_mb, 'md5': actual_md5}
            return dest
        except Exception as e:
            print(f'⚠️ Attempt {attempt}/{retries} failed for {filename}: {e}')
            if tmp.exists(): tmp.unlink()
            if attempt == retries: raise
            time.sleep(3 * attempt)

def extract_safe(archive_path):
    with zipfile.ZipFile(archive_path) as z:
        for m in z.infolist():
            p = Path(m.filename)
            if p.is_absolute() or '..' in p.parts: raise ValueError(f'Unsafe ZIP path: {p}')
        z.extractall(DATA_ROOT)

for cls_name in SELECTED:
    fn, md5_val, _ = CATALOG[cls_name]
    arch = download_timed(fn, md5_val)
    extract_safe(arch)
print('Extraction complete.')

### Step 3: Patient Group Isolation & Cross-Class Patient Leakage Verification

In [ ]:
# =====================================================================
# Cell 3: Patient Grouping & Global Leakage Verification
# =====================================================================
records = []

for class_name in SELECTED:
    candidates = [x for x in DATA_ROOT.glob('*') if x.is_dir() and x.name.lower() == class_name.lower()]
    if not candidates:
        candidates = [x for x in DATA_ROOT.rglob('*') if x.is_dir() and x.name.lower() == class_name.lower()]
    if not candidates: raise RuntimeError(f'Could not find directory for {class_name}')
    disease_dir = candidates[0]
    
    for patient_dir in sorted(disease_dir.iterdir()):
        if not patient_dir.is_dir(): continue
        crops_dir = None
        for sub in patient_dir.iterdir():
            if sub.is_dir() and sub.name.lower() == 'crops':
                crops_dir = sub; break
        if crops_dir is None: continue  # Skip masks & metadata
        
        global_pid = f'{class_name}_{patient_dir.name}'
        imgs = sorted([p for p in crops_dir.iterdir() if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}])
        
        if MAX_CROPS_PER_PATIENT is not None and len(imgs) > MAX_CROPS_PER_PATIENT:
            rng = random.Random(f'{SEED}:{global_pid}')
            imgs = sorted(rng.sample(imgs, MAX_CROPS_PER_PATIENT))
            
        for p in imgs:
            records.append({
                'filepath': str(p),
                'label': class_name,
                'patient_id': global_pid,
                'raw_patient_folder': patient_dir.name
            })

manifest = pd.DataFrame(records)
print('=' * 70)
print('DATASET TELEMETRY & PATIENT BREAKDOWN')
print('=' * 70)
summary = manifest.groupby('label').agg(images=('filepath', 'size'), patients=('patient_id', 'nunique'))
print(summary)

# Patient-folder IDs are scoped to class here. Without an original cross-class person ID,
# cross-class identity overlap cannot be independently excluded.
print('⚠️ Cross-class identity cannot be established from class-prefixed folders alone.')
for cls, row in summary.iterrows():
    if int(row['patients']) < 3:
        raise RuntimeError(f'Not enough patient folders for class {cls}: {int(row["patients"])}; need >=3 for train/val/test.')
    if int(row['patients']) < 10:
        print(f'⚠️ Very few independent patients for {cls}: {int(row["patients"])}. Results will be highly unstable.')

# Stratified Patient-Level Split (70% Train, 15% Val, 15% Test)
rng = np.random.default_rng(SEED)
split_map = {}
for cls in SELECTED:
    cls_groups = np.array(sorted(manifest.loc[manifest.label == cls, 'patient_id'].unique()))
    rng.shuffle(cls_groups)
    n = len(cls_groups)
    n_test = max(1, round(0.15 * n))
    n_val = max(1, round(0.15 * n))
    for g in cls_groups[:n_test]: split_map[g] = 'test'
    for g in cls_groups[n_test:n_test + n_val]: split_map[g] = 'val'
    for g in cls_groups[n_test + n_val:]: split_map[g] = 'train'

manifest['split'] = manifest.patient_id.map(split_map)
assert manifest.groupby('patient_id').split.nunique().max() == 1, 'Patient leakage detected!'

patient_split_counts = manifest.groupby(['label', 'split'])['patient_id'].nunique().unstack(fill_value=0)
patient_split_counts.to_csv(OUT / 'split_patient_counts.csv')
print('\nPATIENT GROUPS PER SPLIT (split_patient_counts.csv):')
print(patient_split_counts)

required_splits = {'train', 'val', 'test'}
assert required_splits.issubset(set(manifest['split'])), 'One or more required splits are empty!'
for cls in SELECTED:
    class_splits = set(manifest.loc[manifest.label == cls, 'split'])
    assert required_splits.issubset(class_splits), f'Missing patient group in at least one split for {cls}'
if (patient_split_counts.min(axis=1) < 2).any():
    print('⚠️ At least one class has fewer than two patients in a split. Avoid clinical conclusions.')

# SHA-256 duplicate verification across splits
seen_hashes = {}
for r in manifest.itertuples(index=False):
    digest = hashlib.sha256(Path(r.filepath).read_bytes()).hexdigest()
    seen_hashes.setdefault(digest, set()).add(r.split)
assert not any(len(splits) > 1 for splits in seen_hashes.values()), 'Exact duplicate image crosses splits!'
print('✅ Image Integrity: Zero duplicate images cross train/val/test splits.')

split_tab = pd.crosstab(manifest.label, manifest.split)
split_tab.to_csv(OUT / 'split_summary.csv')
manifest.to_csv(OUT / 'split_manifest_private.csv', index=False)

### Step 4: Controlled Samplers & Augmentation Pipeline
In **EXP_D**: `WeightedRandomSampler` is active -> Loss function is strictly **unweighted** (prevents double-weighting).

In [ ]:
# =====================================================================
# Cell 4: Controlled Sampling & DataLoader Setup
# =====================================================================
CLASS_NAMES = list(SELECTED)
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

train_t = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
eval_t = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class CropsDataset(Dataset):
    def __init__(self, rows, transform):
        self.rows = rows.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows.iloc[i]
        with Image.open(r.filepath) as im:
            x = self.transform(im.convert('RGB'))
        return x, CLASS_TO_IDX[r.label], r.patient_id

train_df = manifest[manifest.split == 'train'].copy()
val_df = manifest[manifest.split == 'val'].copy()
test_df = manifest[manifest.split == 'test'].copy()

train_ds = CropsDataset(train_df, train_t)
val_ds = CropsDataset(val_df, eval_t)
test_ds = CropsDataset(test_df, eval_t)

counts = train_df.label.value_counts()
class_weights = [len(train_df) / (len(CLASS_NAMES) * counts[c]) for c in CLASS_NAMES]

# --- SAMPLER VS LOSS LOGIC (AVOID DOUBLE WEIGHTING) ---
use_weighted_sampler = EXPERIMENT_MODE in ['EXP_B', 'EXP_D']

if use_weighted_sampler:
    sample_weights = [class_weights[CLASS_TO_IDX[lbl]] for lbl in train_df['label']]
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=DEVICE.type == 'cuda')
    criterion = nn.CrossEntropyLoss()  # Unweighted loss to avoid double-weighting!
    print('⚙️ Sampling Strategy: WeightedRandomSampler active (Loss is standard unweighted).')
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=DEVICE.type == 'cuda')
    weights_tensor = torch.tensor(class_weights, dtype=torch.float, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights_tensor)  # Class-weighted loss
    print(f'⚙️ Sampling Strategy: Standard shuffle active (Loss is class-weighted).')

val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=DEVICE.type == 'cuda')
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=DEVICE.type == 'cuda')

### Step 5: Model Setup (Backbone Freezing & Regularization)
In **EXP_D**: `conv1..layer3` parameters are **frozen**, and BatchNorm modules are explicitly forced into `.eval()` mode.

In [ ]:
# =====================================================================
# Cell 5: Model Initialization & Partial Backbone Freezing
# =====================================================================
model = resnet18(weights=ResNet18_Weights.DEFAULT)

freeze_backbone = EXPERIMENT_MODE in ['EXP_C', 'EXP_D']
if freeze_backbone:
    for name, param in model.named_parameters():
        if not (name.startswith('layer4') or name.startswith('fc')):
            param.requires_grad = False
    dropout_rate = 0.5
    print('🔒 Backbone Regularization: conv1, bn1, layer1-3 FROZEN. Only layer4 and FC trainable.')
else:
    dropout_rate = 0.3
    print('🔓 Backbone Regularization: All ResNet18 layers trainable.')

model.fc = nn.Sequential(
    nn.Dropout(p=dropout_rate),
    nn.Linear(model.fc.in_features, len(CLASS_NAMES))
)
model.to(DEVICE)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f'Trainable Parameters: {trainable_params:,} / {total_params:,} ({trainable_params/total_params*100:.1f}%)')

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

### Step 6: Timed Training Loop with Frozen BatchNorm Eval & Early Stopping

In [ ]:
# =====================================================================
# Cell 6: Timed Training Loop & Validation Monitoring
# =====================================================================
def evaluate(loader):
    model.eval()
    ys, ps, probs, gs = [], [], [], []
    loss_sum, count = 0.0, 0
    with torch.no_grad():
        for xx, yy, grp in loader:
            xx, yy = xx.to(DEVICE), yy.to(DEVICE)
            out = model(xx)
            loss_sum += criterion(out, yy).item() * len(xx)
            count += len(xx)
            ys.extend(yy.cpu().tolist())
            ps.extend(out.argmax(1).cpu().tolist())
            probs.extend(out.softmax(1).cpu().numpy().tolist())
            gs.extend(grp)
    return {'loss': loss_sum / max(count, 1), 'y': ys, 'pred': ps, 'probs': np.array(probs), 'groups': gs}

history = []
best_val_f1 = -1.0
patience_counter = 0
t_training_start = time.perf_counter()

print('=' * 75)
print(f'TRAINING LOG: {EXP_ID} | Max Epochs: {EPOCHS} | Device: {DEVICE}')
print('=' * 75)

for epoch in range(1, EPOCHS + 1):
    t_ep_start = time.perf_counter()
    model.train()
    if freeze_backbone:
        # Enforce eval on frozen BatchNorm layers so running stats do not drift on small batches
        for module_name in ('bn1', 'layer1', 'layer2', 'layer3'):
            for submodule in getattr(model, module_name).modules():
                if isinstance(submodule, nn.modules.batchnorm._BatchNorm):
                    submodule.eval()
                    
    total_loss, total, correct = 0.0, 0, 0
    for x, y, _ in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(x)
        total += len(x)
        correct += (logits.argmax(1) == y).sum().item()
        
    scheduler.step()
    t_ep_end = time.perf_counter()
    ep_duration = round(t_ep_end - t_ep_start, 2)
    
    # Validation pass
    val = evaluate(val_loader)
    val_f1 = f1_score(val['y'], val['pred'], labels=list(range(len(CLASS_NAMES))), average='macro', zero_division=0)
    val_acc = accuracy_score(val['y'], val['pred'])
    
    cm_val = confusion_matrix(val['y'], val['pred'], labels=list(range(len(CLASS_NAMES))))
    recalls = [cm_val[i, i] / max(cm_val[i].sum(), 1) for i in range(len(CLASS_NAMES))]
    
    row = {
        'epoch': epoch,
        'epoch_duration_sec': ep_duration,
        'train_loss': round(total_loss / total, 4),
        'train_acc': round(correct / total, 4),
        'val_loss': round(val['loss'], 4),
        'val_acc': round(val_acc, 4),
        'val_macro_f1': round(val_f1, 4),
        'val_recalls': [round(r, 3) for r in recalls]
    }
    history.append(row)
    pd.DataFrame(history).to_csv(OUT / 'training_history.csv', index=False)
    
    print(f'Epoch [{epoch:02d}/{EPOCHS:02d}] ({ep_duration}s) | '
          f'TrLoss: {row["train_loss"]} Acc: {row["train_acc"]*100:.1f}% || '
          f'ValLoss: {row["val_loss"]} Acc: {row["val_acc"]*100:.1f}% F1: {row["val_macro_f1"]:.4f} '
          f'Recalls: {recalls}')
          
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save({
            'exp_id': EXP_ID,
            'state_dict': model.state_dict(),
            'class_names': CLASS_NAMES,
            'tier': TIER,
            'epoch': epoch,
            'best_f1': best_val_f1
        }, OUT / 'model_best.pth')
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f'⏹️ Early stopping triggered at epoch {epoch} (Validation F1 plateaued).')
            break

total_train_sec = round(time.perf_counter() - t_training_start, 2)
print('=' * 75)
print(f'TRAINING COMPLETE in {total_train_sec} seconds. Best Val Macro F1: {best_val_f1:.4f}')
print('=' * 75)

### Step 7: Final Held-Out Evaluation & Comparison Table

In [ ]:
# =====================================================================
# Cell 7: Final Held-Out Evaluation & Historical Benchmark Comparison
# =====================================================================
ckpt = torch.load(OUT / 'model_best.pth', map_location=DEVICE)
model.load_state_dict(ckpt['state_dict'])

# Use the checkpoint-selected epoch for comparisons, not the final early-stopping epoch.
best_epoch_row = next(h for h in history if h['epoch'] == ckpt['epoch'])
pediculosis_idx = CLASS_TO_IDX.get('pediculosis')
best_val_ped_recall_pct = (100.0 * best_epoch_row['val_recalls'][pediculosis_idx]
                           if pediculosis_idx is not None else float('nan'))
result = evaluate(test_loader)

labels = list(range(len(CLASS_NAMES)))
report = classification_report(result['y'], result['pred'], labels=labels, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
pd.DataFrame(report).transpose().to_csv(OUT / 'classification_report.csv')

print('=' * 70)
print(f'HELD-OUT PATIENT TEST EVALUATION ({EXP_ID})')
print('=' * 70)
print(classification_report(result['y'], result['pred'], labels=labels, target_names=CLASS_NAMES, zero_division=0))

records_pt = []
for grp in sorted(set(result['groups'])):
    inds = [i for i, g in enumerate(result['groups']) if g == grp]
    true_lbl = result['y'][inds[0]]
    avg_probs = result['probs'][inds].mean(0)
    records_pt.append({'group': grp, 'true': true_lbl, 'pred': int(avg_probs.argmax())})
patient_df = pd.DataFrame(records_pt)

metrics = {
    'experiment_id': EXP_ID,
    'experiment_mode': EXPERIMENT_MODE,
    'tier': TIER,
    'classes': CLASS_NAMES,
    'total_training_sec': total_train_sec,
    'best_epoch': ckpt['epoch'],
    'best_val_macro_f1': best_val_f1,
    'best_val_pediculosis_recall': best_val_ped_recall_pct / 100.0 if pediculosis_idx is not None else None,
    'test_crops': len(result['y']),
    'test_patients': len(patient_df),
    'image_accuracy': float(accuracy_score(result['y'], result['pred'])),
    'image_macro_f1': float(f1_score(result['y'], result['pred'], labels=labels, average='macro', zero_division=0)),
    'patient_accuracy': float(accuracy_score(patient_df.true, patient_df.pred)),
    'patient_macro_f1': float(f1_score(patient_df.true, patient_df.pred, labels=labels, average='macro', zero_division=0)),
    'download_telemetry': download_telemetry,
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'scientific_boundary': 'Research benchmark only; significant patient variance must be disclosed'
}
(OUT / 'metrics.json').write_text(json.dumps(metrics, indent=2))

# --- CONTROLLED HISTORICAL COMPARISON TABLE FOR JUDGES ---
historical_data = [
    {'Experiment': 'EXP_A (Baseline v1)', 'Sampler': 'Shuffle', 'Backbone': 'Unfrozen', 'Val Macro F1': '48.37%', 'Val Ped Recall': '5.0%', 'Test Acc': '95.45%'},
    {'Experiment': 'EXP_B (Controlled v2)', 'Sampler': 'WeightedRandom', 'Backbone': 'Unfrozen', 'Val Macro F1': '57.76%', 'Val Ped Recall': '15.0%', 'Test Acc': '92.73%'},
    {'Experiment': f'{EXP_ID}', 'Sampler': 'WeightedRandom', 'Backbone': 'Frozen (conv1..l3)', 'Val Macro F1': f'{best_val_f1*100:.2f}%', 'Val Ped Recall': f'{best_val_ped_recall_pct:.1f}%', 'Test Acc': f'{metrics["image_accuracy"]*100:.2f}%'}
]
comp_df = pd.DataFrame(historical_data)
print('\n' + '=' * 80)
print('DESCRIPTIVE EXPERIMENT HISTORY — NOT FOR MODEL SELECTION USING HELD-OUT TEST')
print('=' * 80)
print(comp_df.to_string(index=False))
comp_df.to_csv(OUT / 'controlled_experiments_comparison.csv', index=False)
print('⚠️ Do not choose EXP_A/B/D by repeated inspection of the same test set; use validation only.')

# Training curves plot
fig, axs = plt.subplots(1, 2, figsize=(14, 5))
hist_df = pd.DataFrame(history)
axs[0].plot(hist_df.epoch, hist_df.train_loss, label='Train Loss', marker='o')
axs[0].plot(hist_df.epoch, hist_df.val_loss, label='Val Loss', marker='s')
axs[0].set_title(f'Loss ({EXP_ID})', fontweight='bold')
axs[0].grid(True, linestyle='--', alpha=0.5); axs[0].legend()

axs[1].plot(hist_df.epoch, hist_df.val_acc * 100, label='Val Accuracy %', marker='o', color='green')
axs[1].plot(hist_df.epoch, hist_df.val_macro_f1 * 100, label='Val Macro F1 %', marker='^', color='orange')
axs[1].set_title('Validation Generalization', fontweight='bold')
axs[1].grid(True, linestyle='--', alpha=0.5); axs[1].legend()
plt.tight_layout()
fig.savefig(OUT / 'training_curves.png', dpi=200)
plt.show()

# Confusion Matrix
cm = confusion_matrix(result['y'], result['pred'], labels=labels)
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(labels); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
ax.set_yticks(labels); ax.set_yticklabels(CLASS_NAMES)
for i in labels:
    for j in labels:
        ax.text(j, i, str(cm[i, j]), ha='center', va='center', color='black' if cm[i, j] < cm.max()/2 else 'white')
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix — {EXP_ID}', fontweight='bold')
plt.tight_layout()
fig.savefig(OUT / 'confusion_matrix.png', dpi=200)
plt.show()

### Step 8: Multi-Column Grad-CAM (Original | Heatmap | Overlay | Prediction)

In [ ]:
# =====================================================================
# Cell 8: Rigorous 3-Column Grad-CAM Layout
# =====================================================================
model.eval()
activations = []
gradients = []
target_layer = model.layer4[-1].conv2

def forward_hook(module, inp, out):
    activations.append(out.detach())
    out.register_hook(lambda g: gradients.append(g.detach()))

hook_handle = target_layer.register_forward_hook(forward_hook)

try:
    # Show representatives from different classes, then fill remaining slots.
    chosen_vis_indices = []
    for class_name in CLASS_NAMES:
        candidates = test_ds.rows.index[test_ds.rows['label'] == class_name].tolist()
        if candidates and len(chosen_vis_indices) < 3:
            chosen_vis_indices.append(candidates[0])
    for candidate in range(len(test_ds)):
        if len(chosen_vis_indices) >= 3: break
        if candidate not in chosen_vis_indices:
            chosen_vis_indices.append(candidate)

    num_vis = len(chosen_vis_indices)
    if num_vis == 0: raise RuntimeError('No held-out test images available')
    fig, axs = plt.subplots(num_vis, 3, figsize=(12, 4 * num_vis))
    if num_vis == 1: axs = [axs]

    for i, sample_index in enumerate(chosen_vis_indices):
        activations.clear(); gradients.clear()
        x, true_y, grp = test_ds[sample_index]
        x_input = x.unsqueeze(0).to(DEVICE).requires_grad_(True)
        
        model.zero_grad(set_to_none=True)
        logits = model(x_input)
        pred_y = int(logits.argmax(1).item())
        conf = float(logits.softmax(1)[0, pred_y].item()) * 100
        logits[0, pred_y].backward()
        
        a = activations[-1]
        g = gradients[-1]
        cam = (g.mean(dim=(2, 3), keepdim=True) * a).sum(1, keepdim=True).relu()
        cam = torch.nn.functional.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)[0, 0].cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        
        raw_img = x.cpu().permute(1, 2, 0).numpy() * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        raw_img = np.clip(raw_img, 0, 1)
        
        axs[i][0].imshow(raw_img); axs[i][0].axis('off')
        axs[i][0].set_title(f'Original Sample {i+1}\nTrue: {CLASS_NAMES[true_y]}', fontsize=10)
        
        axs[i][1].imshow(cam, cmap='jet'); axs[i][1].axis('off')
        axs[i][1].set_title('Grad-CAM Heatmap', fontsize=10)
        
        axs[i][2].imshow(raw_img)
        axs[i][2].imshow(cam, alpha=0.45, cmap='jet')
        axs[i][2].axis('off')
        match_color = 'green' if true_y == pred_y else 'red'
        axs[i][2].set_title(f'Pred: {CLASS_NAMES[pred_y]} ({conf:.1f}%)\nResearch example {i+1}',
                            color=match_color, fontsize=10, fontweight='bold')
        
    plt.suptitle(f'Explainability Attribution (Original | Heatmap | Overlay) — {EXP_ID}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    fig.savefig(OUT / 'gradcam_visualizations.png', dpi=200)
    plt.show()
finally:
    hook_handle.remove()

### Step 9: Model Card & Single-Click Proof Packaging

**Privacy default:** No patient image or original patient path is included in the public ZIP. The saved Grad-CAM figure remains in the local evidence folder for authorized review.


In [ ]:
# =====================================================================
# Cell 9: Model Card & Single-Click Proof Packaging
# =====================================================================
env_info = {
    'experiment_id': EXP_ID,
    'run_id': RUN_ID,
    'experiment_mode': EXPERIMENT_MODE,
    'python_version': sys.version,
    'pytorch_version': torch.__version__,
    'torchvision_version': torchvision.__version__,
    'numpy_version': np.__version__,
    'batch_size': BATCH_SIZE,
    'max_crops_per_patient': MAX_CROPS_PER_PATIENT,
    'early_stopping_patience': EARLY_STOPPING_PATIENCE,
    'gpu_hardware': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'seed': SEED,
    'epochs_trained': len(history),
    'total_training_seconds': total_train_sec,
    'tier': TIER,
    'classes': CLASS_NAMES,
    'download_telemetry': download_telemetry
}
(OUT / 'environment.json').write_text(json.dumps(env_info, indent=2))

model_card_text = f'''# 🏷️ Model Card: {EXP_ID}

- **Experiment Configuration**: {EXPERIMENT_MODE}
- **Backbone Regularization**: {'conv1..layer3 Frozen' if freeze_backbone else 'Fully Trainable'}
- **Sampling**: {'WeightedRandomSampler (Unweighted Loss)' if use_weighted_sampler else 'Class-Weighted CrossEntropy'}
- **Best Validation Macro F1**: {best_val_f1:.4f} (at Epoch {ckpt['epoch']})
- **Test Image Accuracy**: {metrics['image_accuracy']*100:.2f}%
- **Test Image Macro F1**: {metrics['image_macro_f1']:.4f}
- **Test Patient Accuracy**: {metrics['patient_accuracy']*100:.2f}%
- **Test Patient Macro F1**: {metrics['patient_macro_f1']:.4f}
- **Total Elapsed Time**: {total_train_sec} seconds on {env_info['gpu_hardware']}
- **Clinical Boundary**: Research benchmark only. Disclose patient variance and sampling limitations.
'''
(OUT / 'model_card.md').write_text(model_card_text)

public_zip = BASE / f'ShasthoSetu_{EXP_ID}_{RUN_ID}_Proofs.zip'
with zipfile.ZipFile(public_zip, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=4) as z:
    for file in sorted(OUT.iterdir()):
        if not file.is_file(): continue
        if file.name == 'split_manifest_private.csv': continue
        if file.name == 'gradcam_visualizations.png' and not INCLUDE_PATIENT_IMAGES_IN_PUBLIC_ZIP: continue
        z.write(file, arcname=file.name)
zip_path = str(public_zip)

print('=' * 75)
print(f'🏆 PROOF PACKAGE READY: {zip_path}')
print('=' * 75)

try:
    from google.colab import files
    print('⬇️ Downloading proof archive directly to browser...')
    files.download(zip_path)
except Exception as e:
    print(f'Download manually from: {zip_path}')